In [2]:
import re
import time
import requests
from bs4 import BeautifulSoup
from supabase import create_client
import os
import json

SUPABASE_URL = os.getenv("SUPABASE_URL")
SUPABASE_KEY = os.getenv("SUPABASE_KEY")
BASE_URL = "https://asia.pokemon-card.com/ph/event-search/"
CACHE_FILE = "results.json"

supabase = create_client(SUPABASE_URL, SUPABASE_KEY)
session = requests.Session()

headers = {
    "user-agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/145.0.0.0 Safari/537.36"
}


def get_events_with_pending_results() -> list[int]:
    try:
        # Uses an RPC function to do the exclusion entirely in one DB query,
        # avoiding the Supabase JS client's 1000-row default page limit which
        # previously caused events with existing results to be re-scraped.
        response = supabase.rpc("get_events_pending_scrape").execute()

        print(f"🔍 Found {len(response.data)} events pending result scraping.")

        return [r["id"] for r in response.data]
    except Exception as e:
        print(f"❌ Error fetching pending events: {e}")
        return []


def scrape_updated_event_data(event_id: int, soup: BeautifulSoup):
    # FIX #9: Guard against missing venue element to prevent IndexError
    venue_el = soup.select("#place div")
    if not venue_el:
        print(f"⚠️ Could not find venue element for event {event_id}. Skipping venue update.")
        return

    venue = venue_el[-1].get_text(strip=True)
    supabase.table("events").update({"venue": venue}).eq("id", event_id).execute()


def scrape_event_results(event_id: int, soup: BeautifulSoup):
    rows = soup.select(".rankingTable .tableData")

    if not rows:
        print(f"⚠️ No rows found for event ID {event_id}")
        return []

    results_data = []
    for row in rows:
        rank_el = row.select_one(".rank")
        # FIX #2: Strip non-numeric characters (e.g. "1st", "2nd") and cast to int
        # to match the results.rank integer column
        rank_raw = rank_el.get_text(strip=True)
        rank = int(re.sub(r"[^\d]", "", rank_raw))

        points_el = row.select_one(".point")
        points = int(points_el.get_text(strip=True).replace("pt", ""))

        trainer_id_el = row.select_one(".userId")
        trainer_id = trainer_id_el.get_text(strip=True)

        results_data.append(
            {
                "event_id": event_id,
                "rank": rank,
                "trainer_id": trainer_id,
                "points": points,
            }
        )

    return results_data


def scrape_all_events() -> list[dict]:
    all_results = []

    for event_id in get_events_with_pending_results():
        print(f"⏳ Scraping event ID {event_id}")

        try:
            # FIX #5: Add timeout and raise on non-200 responses
            res = session.get(
                f"{BASE_URL}{event_id}", headers=headers, timeout=15
            )
            res.raise_for_status()
        except requests.exceptions.HTTPError as e:
            print(f"❌ HTTP error for event {event_id}: {e}. Skipping.")
            continue
        except requests.exceptions.RequestException as e:
            print(f"❌ Request failed for event {event_id}: {e}. Skipping.")
            continue

        soup = BeautifulSoup(res.text, "html.parser")

        scrape_updated_event_data(event_id, soup)
        results = scrape_event_results(event_id, soup)

        # FIX #7: Removed redundant O(n²) in-loop dedup — dedup is handled
        # later via the unique_results dict
        if results:
            all_results.extend(results)

        # FIX #8: Delay between requests to avoid rate limiting
        time.sleep(1)

    return all_results


def sync_results(final_payload: list[dict]):
    try:
        # FIX #3: Specify on_conflict so Supabase knows to update on (event_id, trainer_id)
        # rather than blindly inserting and creating duplicates
        supabase.table("results").upsert(
            final_payload, on_conflict="event_id,trainer_id"
        ).execute()
        print(f"🚀 Successfully synced {len(final_payload)} results to Supabase.")
    except Exception as e:
        print(f"❌ Supabase Error: {e}")


# FIX #10: Wrap all execution logic in a __main__ guard
if __name__ == "__main__":
    all_results = []

    if os.path.exists(CACHE_FILE):
        # FIX #6: Warn clearly that cached data is being used — this skips
        # get_events_with_pending_results() entirely and upserts whatever
        # is in the file, which may include already-synced results
        print(
            f"📂 {CACHE_FILE} found. Loading from local cache instead of scraping. "
            "Delete this file to force a fresh scrape."
        )
        with open(CACHE_FILE, "r") as f:
            all_results = json.load(f)
    else:
        all_results = scrape_all_events()

        with open(CACHE_FILE, "w") as f:
            json.dump(all_results, f, indent=2)
        print(f"💾 Saved raw results to {CACHE_FILE}")

    # Deduplicate by (event_id, trainer_id), last occurrence wins
    unique_results: dict[tuple, dict] = {}
    for item in all_results:
        key = (item["event_id"], item["trainer_id"])
        unique_results[key] = item

    final_payload = list(unique_results.values())

    with open("results-deduped.json", "w") as f:
        json.dump(final_payload, f, indent=2)
    print(f"💾 Saved deduped results to results-deduped.json ({len(final_payload)} records)")

    if final_payload:
        sync_results(final_payload)


🔍 Found 37 events pending result scraping.
⏳ Scraping event ID 2559
⚠️ No rows found for event ID 2559
⏳ Scraping event ID 2546
⏳ Scraping event ID 3817
⏳ Scraping event ID 3835
⏳ Scraping event ID 3872
⏳ Scraping event ID 3842
⚠️ No rows found for event ID 3842
⏳ Scraping event ID 3986
⏳ Scraping event ID 3957
⏳ Scraping event ID 4098
⏳ Scraping event ID 3967
⏳ Scraping event ID 3961
⏳ Scraping event ID 4215
⏳ Scraping event ID 3959
⏳ Scraping event ID 3998
⏳ Scraping event ID 3857
⏳ Scraping event ID 3843
⏳ Scraping event ID 3827
⏳ Scraping event ID 3897
⚠️ No rows found for event ID 3897
⏳ Scraping event ID 3916
⏳ Scraping event ID 3890
⏳ Scraping event ID 3844
⏳ Scraping event ID 4180
⏳ Scraping event ID 3962
⏳ Scraping event ID 3958
⏳ Scraping event ID 3841
⚠️ No rows found for event ID 3841
⏳ Scraping event ID 3952
⏳ Scraping event ID 3840
⏳ Scraping event ID 3924
⚠️ No rows found for event ID 3924
⏳ Scraping event ID 3834
⚠️ No rows found for event ID 3834
⏳ Scraping event ID 41